# Document Ingestion

This notebook demonstrates how to ingest markdown documents into Pinecone.

In [1]:
from sahiloan_chatbot.application.ingest_documents_service import PineconeIngester
from sahiloan_chatbot.infrastructure.db import get_pinecone_index
from pathlib import Path

## 1. Check Pinecone Connection

In [2]:
# Get index and check stats
index = get_pinecone_index()
stats = index.describe_index_stats()
print(f"Total vectors: {stats['total_vector_count']}")
print(f"Index dimension: {stats['dimension']}")
print(f"Namespaces: {stats.get('namespaces', {})}")

/Users/vishnum/Library/Caches/pypoetry/virtualenvs/sahiloan-chatbot-h6twZ8gO-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total vectors: 25
Index dimension: 1024
Namespaces: {'__default__': {'vector_count': 25}}


## 2. Load FAQ Files

In [4]:
# Use actual FAQ files from data/faqs directory
faq_dir = Path("../data/faqs")

# List all markdown files in the directory
faq_files = list(faq_dir.glob("*.md"))

print(f"Found {len(faq_files)} FAQ files:")
for f in faq_files:
    size = f.stat().st_size
    print(f"  - {f.name} ({size} bytes)")

# Preview content from the files
print("\n" + "="*80)
print("FILE PREVIEWS")
print("="*80)

for faq_file in faq_files:
    content = faq_file.read_text()
    print(f"\n📄 {faq_file.name}")
    print(f"Size: {len(content)} characters")
    print(f"Preview:\n{content[:300]}...")
    print("-"*80)

Found 2 FAQ files:
  - general.md (3970 bytes)
  - sahiloan.md (5951 bytes)

FILE PREVIEWS

📄 general.md
Size: 3929 characters
Preview:

### What exactly is a Home Loan?

A Home Loan is a loan taken to buy, build, or renovate a residential property.
The property itself is kept as security by the bank until the loan is fully repaid.

You repay the loan in monthly EMIs over a long period—usually 15 to 30 years.


### What is a Loan Ag...
--------------------------------------------------------------------------------

📄 sahiloan.md
Size: 5867 characters
Preview:
### How is Sahiloan different from bank relationship managers?

Bank RMs work for banks. We work for you. Our loyalty is to your financial well-being—not monthly targets.

### Why should I trust Sahiloan with such a big decision?

Because we believe trust is built through clarity, not promises.
No h...
--------------------------------------------------------------------------------


## 3. Initialize Document Ingester

In [4]:
# Initialize with custom chunk size
ingester = PineconeIngester(
    chunk_size=500,  # Smaller chunks for testing
    chunk_overlap=100,
    batch_size=10
)

2026-01-26 16:14:01.342 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:__init__:55 - PineconeIngester initialized | chunk_size: 500 | overlap: 100


## 4. Test Chunking (Without Uploading)

In [6]:
# Test chunking on the first FAQ file
test_file = faq_files[0]
content = ingester.read_markdown_file(test_file)
metadata = {"source": str(test_file), "filename": test_file.name, "file_type": "markdown"}
chunks = ingester.chunk_text(content, metadata)

print(f"Testing with: {test_file.name}")
print(f"Content size: {len(content)} characters")
print(f"Number of chunks: {len(chunks)}")
print(f"\n" + "="*80)
print("FIRST CHUNK PREVIEW")
print("="*80)
print(f"Text:\n{chunks[0]['text'][:300]}...")
print(f"\nMetadata: {chunks[0]['metadata']}")

Testing with: general.md
Content size: 3929 characters
Number of chunks: 10

FIRST CHUNK PREVIEW
Text:
### What exactly is a Home Loan?

A Home Loan is a loan taken to buy, build, or renovate a residential property.
The property itself is kept as security by the bank until the loan is fully repaid.

You repay the loan in monthly EMIs over a long period—usually 15 to 30 years.


### What is a Loan Aga...

Metadata: {'source': '../data/faqs/general.md', 'filename': 'general.md', 'file_type': 'markdown', 'chunk_index': 0, 'total_chunks': 10, 'chunk_size': 424}


## 5. Ingest All FAQ Files to Pinecone

In [7]:
# Ingest all FAQ files
print(f"Ingesting {len(faq_files)} FAQ files...\n")

for faq_file in faq_files:
    print(f"📄 Processing: {faq_file.name}")
    ingester.ingest_file(faq_file)
    print()

print("✅ All files ingested successfully!")

2026-01-26 16:08:52.817 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:195 - Ingesting file: ../data/faqs/general.md
2026-01-26 16:08:52.821 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:220 - Created 10 text chunks from ../data/faqs/general.md
2026-01-26 16:08:52.822 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:156 - Generating embeddings for 10 chunks...


Ingesting 2 FAQ files...

📄 Processing: general.md


2026-01-26 16:08:57.052 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:180 - Upserting 10 vectors to Pinecone...
2026-01-26 16:08:58.431 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:184 - Upserted batch 1/1
2026-01-26 16:08:58.432 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:186 - Successfully upserted 10 vectors
2026-01-26 16:08:58.433 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:225 - Successfully ingested ../data/faqs/general.md
2026-01-26 16:08:58.434 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:195 - Ingesting file: ../data/faqs/sahiloan.md
2026-01-26 16:08:58.437 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:220 - Created 15 text chunks from ../data/faqs/sahiloan.md
2026-01-26 16:08:58.438 | INFO     | sahiloan_chatbot.application.inge


📄 Processing: sahiloan.md


2026-01-26 16:09:04.287 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:180 - Upserting 15 vectors to Pinecone...
2026-01-26 16:09:04.913 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:184 - Upserted batch 1/2
2026-01-26 16:09:05.314 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:184 - Upserted batch 2/2
2026-01-26 16:09:05.314 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:upsert_chunks:186 - Successfully upserted 15 vectors
2026-01-26 16:09:05.315 | INFO     | sahiloan_chatbot.application.ingest_documents_service.ingest:ingest_file:225 - Successfully ingested ../data/faqs/sahiloan.md



✅ All files ingested successfully!


## 6. Verify Ingestion

In [8]:
# Check updated stats
stats = index.describe_index_stats()
print("="*80)
print("INGESTION VERIFICATION")
print("="*80)
print(f"Total vectors in Pinecone: {stats['total_vector_count']}")
print(f"Index dimension: {stats['dimension']}")
print(f"Files ingested: {len(faq_files)}")

# Show breakdown by file
print(f"\nNamespaces: {stats.get('namespaces', 'default')}")

INGESTION VERIFICATION
Total vectors in Pinecone: 25
Index dimension: 1024
Files ingested: 2

Namespaces: {'__default__': {'vector_count': 25}}


## 7. Test Semantic Search (Verify Retrieval Works)

In [6]:
# Test queries to verify retrieval works
test_queries = [
    "What is sahiloan",
    "How is EMI calculated?"
]

print("="*80)
print("TESTING RETRIEVAL")
print("="*80)

for query in test_queries:
    print(f"\n🔍 Query: {query}\n")
    
    # Generate embedding for query
    query_embedding = ingester.embeddings.embed_query(query)
    
    # Search in Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=5,
        include_metadata=True
    )
    
    # Display results
    for i, match in enumerate(results['matches'], 1):
        print(f"  Result {i}:")
        print(f"    Score: {match['score']:.4f}")
        print(f"    Source: {match['metadata']['filename']}")
        print(f"    Chunk: {match['metadata']['chunk_index'] + 1}/{match['metadata']['total_chunks']}")
        print(f"    Text: {match['metadata']['text'][:150].strip()}...")
        print()
    
    print("-" * 80)

TESTING RETRIEVAL

🔍 Query: What is sahiloan

  Result 1:
    Score: 0.5486
    Source: sahiloan.md
    Chunk: 2/15
    Text: ### Do I have to pay Sahiloan to use the service?

No. Sahiloan is completely free for borrowers. Our advice is unbiased—we don’t charge you anything...

  Result 2:
    Score: 0.5461
    Source: sahiloan.md
    Chunk: 3/15
    Text: ### What types of loans does Sahiloan help with?

We currently help with:

* Home Loans
* Loan Against Property (LAP)
* Balance Transfer & Top-Up
Spec...

  Result 3:
    Score: 0.5288
    Source: sahiloan.md
    Chunk: 6/15
    Text: ### Will Sahiloan stay with me after loan sanction?

Yes—absolutely. That’s what we’re here for. We continue to support you through:
* Disbursement de...

  Result 4:
    Score: 0.5178
    Source: sahiloan.md
    Chunk: 1/15
    Text: ### How is Sahiloan different from bank relationship managers?

Bank RMs work for banks. We work for you. Our loyalty is to your financial well-being—...

  Result 5:
   